# 🚀 ArquiSys AI - Ollama en Google Colab

Este notebook ejecuta Ollama con qwen2.5:3b y expone el endpoint via ngrok.

## Instrucciones:
1. Runtime → Change runtime type → GPU (T4)
2. Ejecutar todas las celdas en orden
3. Al final copiar la URL de ngrok
4. Actualizar .env en tu sistema local

In [ ]:
#@title 1. Instalar todo desde cero
import subprocess
import os
import time

# Instalar zstd y Ollama
print("📦 Instalando dependencias...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, capture_output=True)
print("📥 Instalando Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True)
print("✅ Ollama instalado")

# Buscar ruta de Ollama
result = subprocess.run("which ollama", shell=True, capture_output=True, text=True)
OLLAMA_PATH = result.stdout.strip() or "/usr/local/bin/ollama"
print(f"📍 Ollama encontrado en: {OLLAMA_PATH}")

In [ ]:
#@title 2. Iniciar Ollama
import subprocess
import time

OLLAMA_PATH = "/usr/local/bin/ollama"

# Iniciar Ollama en background
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(5)
print("✅ Ollama iniciado en puerto 11434")

# Verificar que funciona
result = subprocess.run([OLLAMA_PATH, "list"], capture_output=True, text=True)
print("📋 Modelos disponibles:")
print(result.stdout if result.stdout else "Ninguno aún")

In [ ]:
#@title 3. Descargar modelo qwen2.5:3b
import subprocess

OLLAMA_PATH = "/usr/local/bin/ollama"

print("📥 Descargando modelo qwen2.5:3b (~2GB)...")
result = subprocess.run([OLLAMA_PATH, "pull", "qwen2.5:3b"], capture_output=True, text=True, timeout=600)
if result.returncode == 0:
    print("✅ Modelo descargado!")
else:
    print("⚠️ Estado de descarga:", result.returncode)

# Verificar
result = subprocess.run([OLLAMA_PATH, "list"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
#@title 4. Instalar ngrok y configurar token
import subprocess
import os

if not os.path.exists("/usr/local/bin/ngrok"):
    print("📥 Instalando ngrok...")
    subprocess.run("wget -q https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip", shell=True, capture_output=True)
    subprocess.run("unzip -o ngrok.zip", shell=True, capture_output=True)
    subprocess.run("mv ngrok /usr/local/bin/", shell=True, capture_output=True)
    print("✅ ngrok instalado")
else:
    print("✅ ngrok ya instalado")

# Configurar token
subprocess.run("mkdir -p ~/.ngrok2", shell=True, capture_output=True)
ngrok_dir = os.path.expanduser("~/.ngrok2")
with open(os.path.join(ngrok_dir, "ngrok.yml"), "w") as f:
    f.write("authtoken: cr_3D2ftOD9E86bIfdoD8Z3eYEDplc")
print("✅ Token de ngrok configurado")

In [ ]:
#@title 5. Iniciar ngrok y obtener URL
import subprocess
import time
import urllib.request
import json

# Iniciar ngrok
subprocess.Popen(["/usr/local/bin/ngrok", "http", "11434", "--host-header", "rewrite"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(10)

# Obtener URL
try:
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = json.loads(response.read())
    public_url = data['tunnels'][0]['public_url']
    print("="*50)
    print("🎉 URL PÚBLICA DE OLLAMA:")
    print(public_url)
    print("="*50)
    print("\n📝 COPIA ESTA URL y actualizá tu .env:")
    print(f"OLLAMA_BASE_URL={public_url}")
    print("\n⚠️ Esta URL cambia al reiniciar el notebook")
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Esperando más tiempo...")
    time.sleep(15)
    try:
        response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
        data = json.loads(response.read())
        print("🎉 URL:", data['tunnels'][0]['public_url'])
    except:
        print("❌ No se pudo obtener la URL")

In [ ]:
#@title 6. Verificar funcionamiento y probar Ollama
import urllib.request
import json
import time

try:
    response = urllib.request.urlopen("http://localhost:11434/api/tags", timeout=15)
    models = json.loads(response.read())
    print("✅ Modelos disponibles:")
    for m in models.get('models', []):
        print(f"   - {m['name']}")
except Exception as e:
    print(f"⚠️ {e}")

# Prueba rápida con el modelo
print("\n🧪 Probando el modelo...")
try:
    import urllib.request
    import json
    
    payload = json.dumps({
        "model": "qwen2.5:3b",
        "prompt": "Responde solo con 'Hola, funciono!' en español",
        "stream": False
    }).encode()
    
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
        print("✅ Respuesta del modelo:")
        print(result.get("response", "Sin respuesta"))
    print("\n🎉 Ollama funcionando perfectamente en la nube!")
except Exception as e:
    print(f"⚠️ Error en prueba: {e}")

In [ ]:
#@title 7. Crear archivo .env para Streamlit en Colab
import os

# Obtener URL de ngrok actual
try:
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = json.loads(response.read())
    public_url = data['tunnels'][0]['public_url']
except:
    public_url = "http://localhost:11434"

# Crear .env para Streamlit
env_content = f"""USE_OLLAMA_FOR_ANALYST=true
OLLAMA_MODEL=qwen2.5:3b
OLLAMA_BASE_URL={public_url}
OPENAI_API_KEY=
USE_LLM_FOR_ANALYST=false
"""

with open("/content/.env", "w") as f:
    f.write(env_content)

print("✅ Archivo .env creado en /content/.env")
print(f"OLLAMA_BASE_URL configurado: {public_url}")

In [ ]:
#@title 8. DESCARGAR código de ArquiSys AI desde GitHub
import subprocess

print("📥 Clonando ArquiSys AI desde GitHub...")
subprocess.run("cd /content && rm -rf ArquiSys_Al && git clone https://github.com/VIVA-EL-APRA/ArquiSys_Al.git", 
                shell=True, capture_output=True)
print("✅ Código descargado en /content/ArquiSys_Al")

# Mover .env
subprocess.run("cp /content/.env /content/ArquiSys_Al/.env", shell=True, capture_output=True)
print("✅ .env copiado al proyecto")

In [ ]:
#@title 9. Instalar dependencias y ejecutar Streamlit
import subprocess
import time

print("📦 Instalando dependencias...")
subprocess.run("cd /content/ArquiSys_Al && py -m pip install -e . -q", shell=True, capture_output=True)
subprocess.run("pip install streamlit -q", shell=True, capture_output=True)
print("✅ Dependencias instaladas")

print("\n🚀 Iniciando interfaz web...")
print("="*60)
print("⚠️ Esta celda initiating Streamlit. Cuando veas el output,")
print("buscá la línea que dice 'Network URL' o 'External URL'.")
print("="*60)

# Ejecutar streamlit
subprocess.Popen(
    "cd /content/ArquiSys_Al && streamlit run scripts/ui_app.py --server.port 8501 --server.address 0.0.0.0",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
time.sleep(8)

print("\n✅ Streamlit iniciado!")
print("\n📌 Para acceder a la interfaz:")
print("1. Si ves una URL tipo 'Network URL: http://172.28.0.12:8501'")
print("   -> Agregá un tunnel con el botón de 'Public URL' en Colab")
print("\n2. O ejecutá la celda siguiente para crear un tunnel automático")

In [ ]:
#@title 10. CREAR TUNNEL para Streamlit (hace click en el enlace)
import subprocess
import time
import urllib.request
import json

print("🌐 Creando tunnel para Streamlit...")

# Usar npx con localtunnel para exponer streamlit
subprocess.Popen(
    "npx -y localtunnel --port 8501",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(15)

print("\n" + "="*60)
print("🎉 YA PUEDES ACCEDER A LA INTERFAZ!")
print("="*60)
print("\n📝 El tunnel puede tardar ~30 segundos en iniciar.")
print("Cuando esté listo, vas a ver un enlace abajo.")
print("\n⚠️ IMPORTANTE: Cuando accedas, vas a ver un mensaje de localtunnel")
print("   'Click Continue' para acceder a la interfaz.")
print("\n⌛ Esperando que genere el enlace...")
print("(Si no ves el enlace, ejecutá esta celda de nuevo)")

In [ ]:
#@title 11. Si NO funcionó - Opciones de respaldo
print("""
╔══════════════════════════════════════════════════════════════╗
║              ⚠️ SI NO SE GENERÓ EL ENLACE                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                                      ║
║  OPCIÓN 1: Revisar si Streamlit está corriendo                   ║
║  Ejecutá esta celda para ver el estado:                           ║
║                                                                      ║
    !ps aux | grep streamlit                                        ║
║                                                                      ║
║  OPCIÓN 2: Si Ollama responde, podés usar en tu PC local:        ║
║                                                                      ║
    1. Copiá la URL de Ollama (de la celda 5)                       ║
    2. En tu PC: py -m streamlit run scripts/ui_app.py            ║
    3. Actualizá .env con la URL de ngrok                           ║
║                                                                      ║
║  OPCIÓN 3: Reiniciar todo                                         ║
║  Runtime → Restart runtime → ejecutar celdas 1-10 de nuevo       ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════╝
""")

# Verificar si Streamlit está corriendo
result = subprocess.run("ps aux | grep streamlit | grep -v grep", shell=True, capture_output=True, text=True)
if result.stdout:
    print("✅ Streamlit está corriendo")
    print(result.stdout)
else:
    print("❌ Streamlit NO está corriendo. Ejecutá la celda 9 de nuevo.")

# Intentar con ngrok para streamlit
print("\n🔄 Intentando crear tunnel con ngrok para Streamlit...")
subprocess.Popen(
    ["/usr/local/bin/ngrok", "http", "8501", "--host-header", "rewrite"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True
)
time.sleep(10)

try:
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=10)
    data = json.loads(response.read())
    tunnels = data.get('tunnels', [])
    for t in tunnels:
        if '8501' in t.get('config', {}).get('addr', ''):
            print(f"\n🎉 ENLACE PARA STREAMLIT: {t['public_url']}")
            break
    else:
        print("\n📋 Todos los tunnels disponibles:")
        for t in tunnels:
            print(f"   - {t['public_url']}")
except Exception as e:
    print(f"⚠️ Error: {e}")

print("\n" + "="*60)
print("📋 Alternativa: Usar tunnel de la celda 10 (localtunnel)")
print("="*60)

In [ ]:
#@title 12. Obtener URL directa del tunnel existente (si ngrok ya está corriendo)
import urllib.request
import json

# Intentar obtener la URL del tunnel de ngrok que ya está corriendo
try:
    # Aumentar el timeout
    response = urllib.request.urlopen("http://localhost:4040/api/tunnels", timeout=15)
    data = json.loads(response.read())
    tunnels = data.get('tunnels', [])
    
    print("🌐 Tunnels activos:")
    for t in tunnels:
        url = t.get('public_url', 'N/A')
        print(f"   {url}")
        if '8501' in str(t.get('config', {})):
            print(f"\n🎉 ESTE ES PARA STREAMLIT: {url}")
            break
    
    # Buscar específicamente el de streamlit (puerto 8501)
    streamlit_tunnel = None
    for t in tunnels:
        config = t.get('config', {})
        if config.get('addr') == 'localhost:8501' or '8501' in str(config):
            streamlit_tunnel = t['public_url']
            break
    
    if streamlit_tunnel:
        print(f"\n" + "="*50)
        print(f"🎉 ENLACE PARA LA INTERFAZ: {streamlit_tunnel}")
        print(f"="*50)
    else:
        print("\n⚠️ No se encontró tunnel para Streamlit")
        print("📋 Ejecutá la celda 10 (localtunnel)")
        
except Exception as e:
    print(f"⚠️ No se pudo obtener tunnel: {e}")
    print("\n📋 Ejecutá la celda 10 para crear un nuevo tunnel con localtunnel")